In [ ]:
# Training Attention V=K (RMSNorm + SwiGLU)


import os
import json
import time
import math
from typing import Optional

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -----------------------
# Config 
# -----------------------
DATA_DIR = "/kaggle/input/jklu-en-memap-5gb"   
SINGLEFILE_DIR = "/kaggle/input/jklu-en5gb-singlefile"  
BIN_NAME = "wikipedia_tokens.bin"
META_NAME = "wikipedia_tokens_meta.json"
SINGLEFILE_NAME = "merged_wikipedia.txt"

TRAIN_PERCENT = 0.70        
TRAIN_VAL_SPLIT = 0.70      


SEQ_LEN = 256        
BATCH_SIZE = 2       
GRAD_ACCUM = 1       
EPOCHS = 1
LR = 2e-4
NUM_WORKERS = 0      
PIN_MEMORY = False
SAVE_EVERY_STEPS = 2000
OUTPUT_DIR = "./out_ckpt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS_LOG = os.path.join(OUTPUT_DIR, "metrics.jsonl")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "config.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Chunking when building memmap from text
TOKENIZE_CHUNK_LINES = 1000   # number of lines per chunk to pass to tokenizer

# -----------------------
# Utils: meta & memmap
# -----------------------
def load_meta(meta_path: str) -> dict:
    if not os.path.exists(meta_path):
        return {}
    with open(meta_path, "r", encoding="utf-8") as f:
        return json.load(f)

def infer_dtype_from_meta(meta: dict) -> np.dtype:
    if 'dtype' in meta:
        return np.dtype(meta['dtype'])
    if 'numpy_dtype' in meta:
        return np.dtype(meta['numpy_dtype'])
    return np.int32

def load_memmap_tokens(bin_path: str, meta_path: Optional[str] = None) -> np.memmap:
    meta = load_meta(meta_path) if meta_path and os.path.exists(meta_path) else {}
    dtype = infer_dtype_from_meta(meta)
    if 'num_tokens' in meta:
        length = int(meta['num_tokens'])
    elif 'length' in meta:
        length = int(meta['length'])
    else:
        fsize = os.path.getsize(bin_path)
        length = fsize // dtype.itemsize
    print(f"Opening memmap: {bin_path} (dtype={dtype}, length={length:,})")
    mem = np.memmap(bin_path, dtype=dtype, mode='r', shape=(length,))
    return mem

# -----------------------
# Build memmap from text (chunked, safe)
# -----------------------
def build_memmap_from_text_streaming(text_path: str, out_bin: str, out_meta: str,
                                     tokenizer_name: Optional[str] = None):
    print(" Building memmap from text (streaming chunked tokenizer)...")
    use_hf = False
    tok = None
    if tokenizer_name:
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
            use_hf = True
            print(f"Using HF tokenizer: {tokenizer_name}")
        except Exception:
            print("Could not load HF tokenizer — falling back to byte-level chunking.")
            use_hf = False

    if os.path.exists(out_bin):
        print("Removing existing incomplete bin file:", out_bin)
        os.remove(out_bin)

    total_tokens = 0
    dtype = np.int32

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        chunk_lines = []
        for lineno, line in enumerate(f, 1):
            chunk_lines.append(line)
            if lineno % TOKENIZE_CHUNK_LINES == 0:
                txt = "".join(chunk_lines)
                if use_hf:
                    enc = tok(txt, add_special_tokens=False, return_attention_mask=False)
                    ids = np.array(enc["input_ids"], dtype=np.int32)
                else:
                    b = txt.encode("utf-8", errors="ignore")
                    ids = np.frombuffer(b, dtype=np.uint8).astype(np.int32)
                if ids.size:
                    with open(out_bin, "ab") as wf:
                        wf.write(ids.tobytes())
                    total_tokens += ids.size
                chunk_lines = []
                print(f"  tokenized lines up to {lineno:,}  -> tokens total {total_tokens:,}")
        # final small chunk
        if chunk_lines:
            txt = "".join(chunk_lines)
            if use_hf:
                enc = tok(txt, add_special_tokens=False, return_attention_mask=False)
                ids = np.array(enc["input_ids"], dtype=np.int32)
            else:
                b = txt.encode("utf-8", errors="ignore")
                ids = np.frombuffer(b, dtype=np.uint8).astype(np.int32)
            if ids.size:
                with open(out_bin, "ab") as wf:
                    wf.write(ids.tobytes())
                total_tokens += ids.size
            print(f"  final chunk tokenized -> tokens total {total_tokens:,}")

    meta = {
        "dtype": str(dtype),
        "num_tokens": int(total_tokens),
        "tokenizer": tokenizer_name if tokenizer_name else "byte_level_fallback"
    }
    with open(out_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f" Wrote memmap bin {out_bin} ({total_tokens:,} tokens) and meta {out_meta}.")

# -----------------------
# Dataset: sample context windows from memmap
# -----------------------
class MemmapTokenDataset(Dataset):
    def __init__(self, memmap: np.memmap, seq_len: int, split: str = "train", split_ratio=0.98):
        assert split in ("train", "val")
        N = memmap.shape[0]
        split_idx = int(N * split_ratio)
        if split == "train":
            self.start = 0
            self.end = split_idx
        else:
            self.start = split_idx
            self.end = N
        self.seq_len = seq_len
        self.memmap = memmap
        self.num_examples = max(0, (self.end - self.start) - seq_len)
    def __len__(self):
        return self.num_examples
    def __getitem__(self, idx):
        pos = self.start + idx
        seq = np.array(self.memmap[pos: pos + self.seq_len + 1], dtype=np.int64)  # +1 for target shift
        input_ids = torch.from_numpy(seq[:-1]).long()
        target_ids = torch.from_numpy(seq[1:]).long()
        return input_ids, target_ids

def collate_batch(batch):
    inputs = torch.stack([b[0] for b in batch], dim=0)
    targets = torch.stack([b[1] for b in batch], dim=0)
    return inputs, targets

# -----------------------
# Model components: RMSNorm, SwiGLU, Attention (V=K)
# -----------------------
class RMSNormSimple(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

class SwiGLUFFN_simple(nn.Module):
    def __init__(self, d_model: int, expansion: int = 4, dropout: float = 0.1):
        super().__init__()
        hidden = d_model * expansion
        self.project_in = nn.Linear(d_model, hidden * 2)
        self.project_out = nn.Linear(hidden, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x_proj = self.project_in(x)
        a, b = x_proj.chunk(2, dim=-1)
        x_ff = a * torch.nn.functional.silu(b)
        x_out = self.project_out(x_ff)
        return self.dropout(x_out)

class CausalKVAttention_simple(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = (self.head_dim) ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
    def _shape(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    def _unshape(self, x):
        B, nh, T, hd = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, nh * hd)
    def forward(self, x, cache_k: Optional[torch.Tensor] = None):
        B, T, D = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        qh = self._shape(q)
        kh = self._shape(k)
        vh = kh  # V = K
        if cache_k is not None:
            kh = torch.cat([cache_k, kh], dim=2)
            vh = kh
        T_total = kh.size(2)
        scores = torch.matmul(qh, kh.transpose(-2, -1)) * self.scale
        device = x.device
        causal_mask = torch.triu(torch.ones((T, T_total), dtype=torch.bool, device=device), diagonal=1)
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = torch.nn.functional.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        ctx = torch.matmul(attn, vh)
        ctx = self._unshape(ctx)
        out = self.out_proj(ctx)
        out = self.proj_dropout(out)
        new_cache_k = kh if cache_k is not None else None
        return out, new_cache_k

class GPTBlockKVasK_simple(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_expansion: int = 4,
                 attn_dropout: float = 0.1, ffn_dropout: float = 0.1):
        super().__init__()
        self.rms1 = RMSNormSimple(d_model)
        self.attn = CausalKVAttention_simple(d_model, n_heads, dropout=attn_dropout)
        self.rdrop1 = nn.Dropout(attn_dropout)
        self.rms2 = RMSNormSimple(d_model)
        self.ffn = SwiGLUFFN_simple(d_model, expansion=ffn_expansion, dropout=ffn_dropout)
        self.rdrop2 = nn.Dropout(ffn_dropout)
    def forward(self, x, cache_k: Optional[torch.Tensor] = None):
        x_norm = self.rms1(x)
        attn_out, new_cache_k = self.attn(x_norm, cache_k=cache_k)
        x = x + self.rdrop1(attn_out)
        x_norm = self.rms2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.rdrop2(ffn_out)
        return x, new_cache_k

class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 512, n_heads: int = 8, n_layers: int = 6,
                 seq_len: int = 256, ffn_expansion: int = 4, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, seq_len, d_model))
        self.blocks = nn.ModuleList([GPTBlockKVasK_simple(d_model, n_heads, ffn_expansion, dropout, dropout)
                                     for _ in range(n_layers)])
        self.ln_f = RMSNormSimple(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.seq_len = seq_len
    def forward(self, input_ids, cache_k: Optional[torch.Tensor] = None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        x = x + self.pos_emb[:, :T, :]
        new_cache = None
        for blk in self.blocks:
            x, new_cache = blk(x, cache_k=cache_k)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, new_cache

# -----------------------
# Helpers: save checkpoint & append metrics
# -----------------------
def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer, global_step: int, extra: dict):
    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "global_step": global_step,
        "extra": extra
    }
    torch.save(payload, path)
    config = extra.get("config", {})
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

def append_metrics(epoch: int, global_step: int, val_loss: float, perplexity: float):
    rec = {
        "timestamp": time.time(),
        "epoch": epoch,
        "global_step": global_step,
        "val_loss": float(val_loss),
        "perplexity": float(perplexity)
    }
    with open(METRICS_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")

# -----------------------
# Training function: memmap prints + safe loop with AMP & OOM handling
# -----------------------
def train():
    print("\n==============================")
    print(" Checking dataset & memmap files")
    print("==============================\n")

    bin_path = os.path.join(DATA_DIR, BIN_NAME)
    meta_path = os.path.join(DATA_DIR, META_NAME)
    text_path = os.path.join(SINGLEFILE_DIR, SINGLEFILE_NAME)

    print(f"Looking for bin: {bin_path}")
    print(f"Looking for meta: {meta_path}")

    if not os.path.exists(bin_path):
        print(" Memmap .bin not found.")
        if os.path.exists(text_path):
            print(f"Found fallback text: {text_path}")
            build_memmap_from_text_streaming(text_path, bin_path, meta_path, tokenizer_name=None)
        else:
            raise FileNotFoundError(f"Neither {bin_path} nor {text_path} found. Provide dataset.")

    # load memmap
    mem = load_memmap_tokens(bin_path, meta_path)
    total_tokens = len(mem)
    print(f" Loaded memmap: shape={mem.shape}, dtype={mem.dtype}, total tokens={total_tokens:,}")

    # -------------------------------
    # Limit dataset to TRAIN_PERCENT
    # -------------------------------
    MAX_TOKENS = int(total_tokens * TRAIN_PERCENT)
    print(f" Using only first {TRAIN_PERCENT*100:.0f}% of data → {MAX_TOKENS:,} tokens")
    mem = mem[:MAX_TOKENS]

    meta = load_meta(meta_path) if os.path.exists(meta_path) else {}
    vocab_size = meta.get("vocab_size", None)
    if vocab_size is None:
        try:
            vocab_size = int(np.max(mem)) + 1
            print(f"Inferred vocab_size: {vocab_size}")
        except Exception:
            vocab_size = 65536
            print(f"Could not infer vocab_size. Defaulting to {vocab_size}")

    print("\n Creating datasets (on sliced memmap)...")
    train_ds = MemmapTokenDataset(mem, seq_len=SEQ_LEN, split="train", split_ratio=TRAIN_VAL_SPLIT)
    val_ds = MemmapTokenDataset(mem, seq_len=SEQ_LEN, split="val", split_ratio=TRAIN_VAL_SPLIT)
    print(f" Train samples: {len(train_ds):,}")
    print(f" Val samples  : {len(val_ds):,}")

    print("\n Building DataLoaders...")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_batch, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_batch, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print("DataLoaders ready.")

    print("\n Initializing model...")
    model = MiniGPT(vocab_size=vocab_size, d_model=512, n_heads=8, n_layers=6, seq_len=SEQ_LEN).to(DEVICE)
    print("Model on device:", DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

    # write config file (initial)
    config = {
        "DATA_DIR": DATA_DIR,
        "BIN_NAME": BIN_NAME,
        "META_NAME": META_NAME,
        "TRAIN_PERCENT": TRAIN_PERCENT,
        "TRAIN_VAL_SPLIT": TRAIN_VAL_SPLIT,
        "SEQ_LEN": SEQ_LEN,
        "BATCH_SIZE": BATCH_SIZE,
        "GRAD_ACCUM": GRAD_ACCUM,
        "EPOCHS": EPOCHS,
        "LR": LR,
        "NUM_WORKERS": NUM_WORKERS,
        "DEVICE": DEVICE,
        "vocab_size": int(vocab_size)
    }
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    global_step = 0
    model.train()

    for epoch in range(EPOCHS):
        print("\n=====================================")
        print(f" Starting Epoch {epoch+1}/{EPOCHS}")
        print("=====================================\n")
        epoch_start = time.time()
        steps_per_epoch = len(train_loader)
        print(f"Steps this epoch: {steps_per_epoch}, BATCH_SIZE: {BATCH_SIZE}, SEQ_LEN: {SEQ_LEN}")

        for batch_idx, (input_ids, target_ids) in enumerate(train_loader):
            # move to device
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            target_ids = target_ids.to(DEVICE, non_blocking=True)

            # forward / backward with AMP
            try:
                with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                    logits, _ = model(input_ids)
                    loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
                    loss = loss / GRAD_ACCUM

                scaler.scale(loss).backward()

                if (batch_idx + 1) % GRAD_ACCUM == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    global_step += 1

                # periodic prints
                if global_step % 50 == 0:
                    print(f"[Epoch {epoch+1}] Global step {global_step} | batch {batch_idx+1}/{steps_per_epoch} | loss {loss.item()*GRAD_ACCUM:.6f}")

            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    print(f" OOM at batch {batch_idx+1} — clearing grad/cache and skipping step.")
                    optimizer.zero_grad()
                    if DEVICE == "cuda":
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise

            # ETA every 200 batch reads
            if (batch_idx + 1) % 200 == 0:
                elapsed = time.time() - epoch_start
                steps_done = batch_idx + 1
                avg_step = elapsed / max(1, steps_done)
                eta = (steps_per_epoch - steps_done) * avg_step
                print(f" ETA for epoch: {eta/60:.2f} minutes")

            # checkpointing
            if global_step and global_step % SAVE_EVERY_STEPS == 0:
                ckpt_path = os.path.join(OUTPUT_DIR, f"ckpt_step{global_step}.pt")
                save_checkpoint(ckpt_path, model, optimizer, global_step, {"config": config})
                print(" Saved checkpoint ->", ckpt_path)
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

        epoch_time = time.time() - epoch_start
        print(f"\n Epoch {epoch+1} finished in {epoch_time/60:.2f} minutes")

        # validation (small sample) — compute average loss and perplexity
        print("\n Running quick validation sample...")
        model.eval()
        with torch.no_grad():
            val_losses = []
            for i, (input_ids, target_ids) in enumerate(val_loader):
                input_ids = input_ids.to(DEVICE, non_blocking=True)
                target_ids = target_ids.to(DEVICE, non_blocking=True)
                logits, _ = model(input_ids)
                loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
                val_losses.append(loss.item())
                if i >= 50:  # sample up to 51 batches for a reasonable estimate (adjustable)
                    break
            if val_losses:
                avg_val_loss = float(sum(val_losses) / len(val_losses))
                # perplexity = exp(avg_val_loss)
                try:
                    perp = float(math.exp(avg_val_loss))
                    # protect against overflow
                    if math.isinf(perp) or perp > 1e30:
                        perp = float('inf')
                except OverflowError:
                    perp = float('inf')
                print(f" Validation loss (sample of {len(val_losses)} batches): {avg_val_loss:.5f}")
                print(f" Perplexity (sample): {perp:.3f}")
                # append metrics to log file
                append_metrics(epoch + 1, global_step, avg_val_loss, perp)
                # save checkpoint for this epoch (with metrics)
                ckpt_path = os.path.join(OUTPUT_DIR, f"ckpt_epoch{epoch+1}_step{global_step}.pt")
                extra = {"config": config, "val_loss": avg_val_loss, "perplexity": perp, "epoch": epoch + 1}
                save_checkpoint(ckpt_path, model, optimizer, global_step, extra)
                print(" Saved epoch checkpoint ->", ckpt_path)
        model.train()

    # final save + config and metrics
    final_path = os.path.join(OUTPUT_DIR, "final_ckpt.pt")
    save_checkpoint(final_path, model, optimizer, global_step, {"config": config})
    print("Training complete. Saved final checkpoint ->", final_path)
    print("Config saved to:", CONFIG_PATH)
    print("Metrics log appended to:", METRICS_LOG)

if __name__ == "__main__":
    train()


In [2]:
# ============================
# ONE CELL INFERENCE
# ============================


import os
import json
import time
import torch
from torch import nn
import numpy as np

from typing import Optional

# -----------------------
# Paths
# -----------------------
CHECKPOINT_PATH = "/kaggle/input/46k-itr/ckpt_step46000.pt"
CONFIG_PATH = "/kaggle/input/confic/confic.json"
META_PATH = "/kaggle/input/jklu-en-memap-5gb/wikipedia_tokens_meta.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT = "The history of science is"

MAX_NEW_TOKENS = 100
TEMPERATURE = 0.8
TOP_K = 50

# -----------------------
# Model components (same as training)
# -----------------------
class RMSNormSimple(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

class SwiGLUFFN_simple(nn.Module):
    def __init__(self, d_model: int, expansion: int = 4, dropout: float = 0.1):
        super().__init__()
        hidden = d_model * expansion
        self.project_in = nn.Linear(d_model, hidden * 2)
        self.project_out = nn.Linear(hidden, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x_proj = self.project_in(x)
        a, b = x_proj.chunk(2, dim=-1)
        x_ff = a * torch.nn.functional.silu(b)
        x_out = self.project_out(x_ff)
        return self.dropout(x_out)

class CausalKVAttention_simple(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = (self.head_dim) ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
    def _shape(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    def _unshape(self, x):
        B, nh, T, hd = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, nh * hd)
    def forward(self, x, cache_k: Optional[torch.Tensor] = None):
        B, T, D = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        qh = self._shape(q)
        kh = self._shape(k)
        vh = kh  # V = K
        if cache_k is not None:
            kh = torch.cat([cache_k, kh], dim=2)
            vh = kh
        T_total = kh.size(2)
        scores = torch.matmul(qh, kh.transpose(-2, -1)) * self.scale
        device = x.device
        causal_mask = torch.triu(torch.ones((T, T_total), dtype=torch.bool, device=device), diagonal=1)
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = torch.nn.functional.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        ctx = torch.matmul(attn, vh)
        ctx = self._unshape(ctx)
        out = self.out_proj(ctx)
        out = self.proj_dropout(out)
        new_cache_k = kh if cache_k is not None else None
        return out, new_cache_k

class GPTBlockKVasK_simple(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_expansion: int = 4,
                 attn_dropout: float = 0.1, ffn_dropout: float = 0.1):
        super().__init__()
        self.rms1 = RMSNormSimple(d_model)
        self.attn = CausalKVAttention_simple(d_model, n_heads, dropout=attn_dropout)
        self.rdrop1 = nn.Dropout(attn_dropout)
        self.rms2 = RMSNormSimple(d_model)
        self.ffn = SwiGLUFFN_simple(d_model, expansion=ffn_expansion, dropout=ffn_dropout)
        self.rdrop2 = nn.Dropout(ffn_dropout)
    def forward(self, x, cache_k: Optional[torch.Tensor] = None):
        x_norm = self.rms1(x)
        attn_out, new_cache_k = self.attn(x_norm, cache_k=cache_k)
        x = x + self.rdrop1(attn_out)
        x_norm = self.rms2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.rdrop2(ffn_out)
        return x, new_cache_k

class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 512, n_heads: int = 8, n_layers: int = 6,
                 seq_len: int = 256, ffn_expansion: int = 4, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, seq_len, d_model))
        self.blocks = nn.ModuleList([GPTBlockKVasK_simple(d_model, n_heads, ffn_expansion, dropout, dropout)
                                     for _ in range(n_layers)])
        self.ln_f = RMSNormSimple(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.seq_len = seq_len
    def forward(self, input_ids, cache_k: Optional[torch.Tensor] = None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        x = x + self.pos_emb[:, :T, :]
        new_cache = None
        for blk in self.blocks:
            x, new_cache = blk(x, cache_k=cache_k)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, new_cache

# -----------------------
# Load checkpoint
# -----------------------
def load_checkpoint(ckpt_path: str, device: str = "cpu"):
    print(f"Loading checkpoint from: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    return checkpoint

# -----------------------
# Tokenizer support: tiktoken, HuggingFace, or byte-level
# -----------------------
_tokenizer_cache = None

def get_tokenizer(tokenizer_type: str):
    """Get or create tokenizer (cached)"""
    global _tokenizer_cache
    if _tokenizer_cache is not None:
        return _tokenizer_cache
    
    if tokenizer_type == "byte_level":
        _tokenizer_cache = None
        return None
    
    # Try tiktoken first (for GPT-2, GPT-3 style tokenizers)
    if "tiktoken" in tokenizer_type.lower() or "gpt" in tokenizer_type.lower():
        try:
            import tiktoken
            # tiktoken_gpt2 maps to gpt2 encoding
            encoding_name = "gpt2"  # or "cl100k_base" for GPT-3.5/4
            enc = tiktoken.get_encoding(encoding_name)
            _tokenizer_cache = ("tiktoken", enc)
            print(f" Loaded tiktoken encoder: {encoding_name}")
            return _tokenizer_cache
        except ImportError:
            print(" tiktoken not installed. Install with: pip install tiktoken")
        except Exception as e:
            print(f" Could not load tiktoken: {e}")
    
    # Try HuggingFace tokenizer
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(tokenizer_type, use_fast=True)
        _tokenizer_cache = ("hf", tok)
        print(f" Loaded HF tokenizer: {tokenizer_type}")
        return _tokenizer_cache
    except Exception as e:
        print(f" Could not load HF tokenizer: {e}")
    
    _tokenizer_cache = None
    return None

def encode_text(text: str, tokenizer_type: str = "byte_level") -> list:
    """Convert text to token IDs"""
    if tokenizer_type == "byte_level":
        return list(text.encode("utf-8", errors="ignore"))
    
    tok = get_tokenizer(tokenizer_type)
    if tok is None:
        print(" Falling back to byte-level encoding")
        return list(text.encode("utf-8", errors="ignore"))
    
    tok_type, tok_obj = tok
    if tok_type == "tiktoken":
        return tok_obj.encode(text)
    elif tok_type == "hf":
        return tok_obj.encode(text, add_special_tokens=False)
    else:
        return list(text.encode("utf-8", errors="ignore"))

def decode_tokens(tokens: list, tokenizer_type: str = "byte_level") -> str:
    """Convert token IDs back to text"""
    if tokenizer_type == "byte_level":
        try:
            return bytes(tokens).decode("utf-8", errors="ignore")
        except:
            return ""
    
    tok = get_tokenizer(tokenizer_type)
    if tok is None:
        try:
            return bytes(tokens).decode("utf-8", errors="ignore")
        except:
            return ""
    
    tok_type, tok_obj = tok
    if tok_type == "tiktoken":
        return tok_obj.decode(tokens)
    elif tok_type == "hf":
        return tok_obj.decode(tokens)
    else:
        try:
            return bytes(tokens).decode("utf-8", errors="ignore")
        except:
            return ""

# -----------------------
# Generation with temperature and top-k sampling
# -----------------------
@torch.no_grad()
def generate(model, prompt_tokens: list, max_new_tokens: int = 100, 
             temperature: float = 1.0, top_k: int = 0, device: str = "cpu"):
    """Generate text autoregressively from prompt"""
    model.eval()
    
    # Convert prompt to tensor
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long).to(device)
    generated = prompt_tokens.copy()
    
    for _ in range(max_new_tokens):
        # Truncate if exceeds seq_len
        if input_ids.size(1) > model.seq_len:
            input_ids = input_ids[:, -model.seq_len:]
        
        # Forward pass
        logits, _ = model(input_ids)
        
        # Get logits for last token
        logits = logits[:, -1, :] / temperature
        
        # Apply top-k filtering
        if top_k > 0:
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = float('-inf')
        
        # Sample from distribution
        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Append to sequence
        generated.append(next_token.item())
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
    
    return generated

# -----------------------
# Load Config
# -----------------------
with open(CONFIG_PATH) as f:
    config = json.load(f)

vocab_size = config["vocab_size"]
seq_len = config["SEQ_LEN"]

tokenizer_type = "byte_level"

if os.path.exists(META_PATH):
    with open(META_PATH) as f:
        meta = json.load(f)
        tokenizer_type = meta.get("tokenizer", "byte_level")

# -----------------------
# Build Model
# -----------------------
model = MiniGPT(
    vocab_size=vocab_size,
    d_model=512,
    n_heads=8,
    n_layers=6,
    seq_len=seq_len,
    ffn_expansion=4,
    dropout=0.1,
).to(DEVICE)

# -----------------------
# Load Checkpoint
# -----------------------
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Model Loaded Successfully!")

# -----------------------
# Parameter Count
# -----------------------
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameters : {total_params:,}")
print(f"Trainable  : {trainable_params:,}")

# -----------------------
# Encode Prompt
# -----------------------
prompt_tokens = encode_text(PROMPT, tokenizer_type)

print(f"\nPrompt : {PROMPT}")
print(f"Prompt Tokens : {len(prompt_tokens)}")

# -----------------------
# Generate
# -----------------------
start = time.time()

generated_tokens = generate(
    model,
    prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    device=DEVICE,
)

end = time.time()

generated_text = decode_tokens(generated_tokens, tokenizer_type)

generated_count = len(generated_tokens) - len(prompt_tokens)

# -----------------------
# Results
# -----------------------
print("\n================ OUTPUT ================\n")
print(generated_text)

print("\n============= STATISTICS =============")
print(f"Prompt Tokens     : {len(prompt_tokens)}")
print(f"Generated Tokens  : {generated_count}")
print(f"Total Tokens      : {len(generated_tokens)}")
print(f"Generation Time   : {end-start:.2f} sec")
print(f"Tokens / Second   : {generated_count/(end-start):.2f}")

print("\nCheckpoint Step :", checkpoint.get("global_step"))

if "extra" in checkpoint:
    print("Validation Loss :", checkpoint["extra"].get("val_loss"))
    print("Perplexity      :", checkpoint["extra"].get("perplexity"))

Model Loaded Successfully!

Parameters : 75,230,720
Trainable  : 75,230,720
 Loaded tiktoken encoder: gpt2

Prompt : The history of science is
Prompt Tokens : 5

================ OUTPUT ================

The history of science is the only person (and others) have been studied within the same-sex couples.

the new age distribution is in the united states in the united states and the united states.

the jaguar county was named after the johann joseph peter i of the u.s. census bureau.

lake rossie county has a total area of , of which is land and is water.

as of the census of 2010, there were 2,

============= STATISTICS =============
Prompt Tokens     : 5
Generated Tokens  : 100
Total Tokens      : 105
Generation Time   : 0.67 sec
Tokens / Second   : 149.17

Checkpoint Step : 46000
Validation Loss : None
Perplexity      : None
